# Experiment: Round 1 EDA

Objective:
- Mirror the tutorial notebook workflow on `data/round 1`.
- Inspect price and trade behaviour for `ASH_COATED_OSMIUM` and `INTARIAN_PEPPER_ROOT`.
- Use `key info.md` as structural guidance for fixed-fair versus dynamic-fair research.


In [ ]:
# Setup: imports, paths, and plotting defaults
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from imc_eda.round1 import DEFAULT_DATA_DIR, build_price_features, load_prices, load_trades

sns.set_theme(style='whitegrid', context='notebook')
FIGURES_DIR = ROOT / 'reports' / 'figures' / 'round1'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RESIN_CANDIDATE = 'ASH_COATED_OSMIUM'
KELP_CANDIDATE = 'INTARIAN_PEPPER_ROOT'
RESIN_FAIR = 10_000

ROOT, DEFAULT_DATA_DIR

## Plan

Hypotheses to test:
- `ASH_COATED_OSMIUM` may behave like the fixed-fair product described in `key info.md`, so deviations around 10,000 should be limited and mispriced quotes should stand out.
- `INTARIAN_PEPPER_ROOT` may behave more like the dynamic-fair product, so a book-derived fair estimate such as `wall_mid` may be more informative than a naive midpoint alone.
- Spread, imbalance, and trade flow should differ enough across products to justify different trader logic.


In [ ]:
# Load Round 1 data and confirm the inventory
prices = build_price_features(load_prices(file_format='csv'))
trades = load_trades(file_format='csv')

inventory = pd.DataFrame(
    {
        'frame': ['prices', 'trades'],
        'rows': [len(prices), len(trades)],
        'days': [prices['day'].nunique(), trades['day'].nunique()],
        'products': [
            sorted(prices['product'].unique().tolist()),
            sorted(trades['symbol'].unique().tolist()),
        ],
    }
)

raw_mid_quality = (
    prices.groupby('product')
    .agg(
        raw_mid_zero_share=('mid_price', lambda s: float((s == 0).mean())),
        book_mid_missing_share=('book_mid', lambda s: float(s.isna().mean())),
        wall_mid_missing_share=('wall_mid', lambda s: float(s.isna().mean())),
    )
    .sort_index()
)

display(inventory)
display(raw_mid_quality.round(4))
display(prices.head())
display(trades.head())

In [ ]:
# Build quick summary tables for first-pass EDA
price_summary = (
    prices.groupby('product')
    .agg(
        observations=('book_mid', 'size'),
        valid_book_mid=('book_mid', lambda s: int(s.notna().sum())),
        mean_book_mid=('book_mid', 'mean'),
        book_mid_std=('book_mid', 'std'),
        mean_spread=('spread', 'mean'),
        spread_p90=('spread', lambda s: s.quantile(0.9)),
        mean_book_imbalance=('book_imbalance', 'mean'),
        mean_wall_minus_book_mid=('wall_mid_minus_book_mid', 'mean'),
    )
    .sort_index()
)

trade_summary = (
    trades.groupby('symbol')
    .agg(
        trades=('price', 'size'),
        average_trade_price=('price', 'mean'),
        total_quantity=('quantity', 'sum'),
        total_notional=('notional', 'sum'),
        average_trade_size=('quantity', 'mean'),
    )
    .sort_index()
)

display(price_summary.round(3))
display(trade_summary.round(3))

In [ ]:
# Build a session-time axis so multi-day charts read as one timeline
quote_price_plot = (
    prices.groupby(['day', 'timestamp', 'product'], as_index=False)[['bid_price_1', 'bid_price_2', 'ask_price_1', 'ask_price_2', 'book_mid', 'wall_mid']]
    .mean()
    .sort_values(['day', 'timestamp', 'product'])
)

timestamp_step = quote_price_plot.groupby('day')['timestamp'].diff().dropna().loc[lambda s: s > 0].min()
day_span = quote_price_plot['timestamp'].max() + (timestamp_step if pd.notna(timestamp_step) else 100)
quote_price_plot['session_time'] = (
    (quote_price_plot['day'] - quote_price_plot['day'].min()) * day_span + quote_price_plot['timestamp']
)

quote_price_long = quote_price_plot.melt(
    id_vars=['day', 'timestamp', 'product', 'session_time'],
    value_vars=['bid_price_1', 'bid_price_2', 'ask_price_1', 'ask_price_2'],
    var_name='quote_level',
    value_name='quote_price',
).dropna(subset=['quote_price'])

products = sorted(quote_price_plot['product'].unique())
day_starts = quote_price_plot.groupby('day')['session_time'].min().sort_index()

fig, axes = plt.subplots(len(products), 1, figsize=(14, 8), sharex=True, constrained_layout=True)
axes = [axes] if len(products) == 1 else axes
for ax, product in zip(axes, products):
    product_data = quote_price_long[quote_price_long['product'] == product]
    sns.lineplot(
        data=product_data,
        x='session_time',
        y='quote_price',
        hue='quote_level',
        marker='o',
        markersize=3,
        linewidth=1.5,
        ax=ax,
    )
    for boundary in day_starts.iloc[1:]:
        ax.axvline(boundary, color='0.75', linestyle='--', linewidth=1)
    ax.set_title(f'{product}: bid and ask prices vs time')
    ax.set_ylabel('Quote price')
    ax.legend(title='Quote level')

axes[-1].set_xticks(day_starts.tolist())
axes[-1].set_xticklabels([f'day {day}' for day in day_starts.index])
axes[-1].set_xlabel('Trading session time')

quote_figure_path = FIGURES_DIR / 'quote-prices-vs-time-separate-assets.png'
fig.savefig(quote_figure_path, dpi=150, bbox_inches='tight')
plt.show()
quote_figure_path

In [ ]:
# Zoom the first 75 observations for each asset to make the early quote ladder easier to inspect
first_n = 75
first_n_quote_price_plot = quote_price_plot.groupby('product', group_keys=False).head(first_n).copy()
first_n_quote_price_long = quote_price_long.groupby(['product', 'quote_level'], group_keys=False).head(first_n).copy()
first_n_day_starts = first_n_quote_price_plot.groupby('day')['session_time'].min().sort_index()

fig, axes = plt.subplots(len(products), 1, figsize=(14, 8), sharex=True, constrained_layout=True)
axes = [axes] if len(products) == 1 else axes
for ax, product in zip(axes, products):
    product_data = first_n_quote_price_long[first_n_quote_price_long['product'] == product]
    sns.lineplot(
        data=product_data,
        x='session_time',
        y='quote_price',
        hue='quote_level',
        marker='o',
        markersize=3,
        linewidth=1.5,
        ax=ax,
    )
    for boundary in first_n_day_starts.iloc[1:]:
        ax.axvline(boundary, color='0.75', linestyle='--', linewidth=1)
    ax.set_title(f'{product}: bid and ask prices vs time (first {first_n} observations)')
    ax.set_ylabel('Quote price')
    ax.legend(title='Quote level')

axes[-1].set_xticks(first_n_day_starts.tolist())
axes[-1].set_xticklabels([f'day {day}' for day in first_n_day_starts.index])
axes[-1].set_xlabel('Trading session time')

first_n_figure_path = FIGURES_DIR / 'quote-prices-vs-time-separate-assets-first-75.png'
fig.savefig(first_n_figure_path, dpi=150, bbox_inches='tight')
plt.show()
first_n_figure_path

In [ ]:
# Plot trade prices against time in separate panels for each asset
trade_price_plot = trades.rename(columns={'symbol': 'product'}).sort_values(['day', 'timestamp', 'product', 'price']).copy()
trade_timestamp_step = trade_price_plot.groupby('day')['timestamp'].diff().dropna().loc[lambda s: s > 0].min()
trade_day_span = trade_price_plot['timestamp'].max() + (trade_timestamp_step if pd.notna(trade_timestamp_step) else 100)
trade_price_plot['session_time'] = (
    (trade_price_plot['day'] - trade_price_plot['day'].min()) * trade_day_span + trade_price_plot['timestamp']
)
trade_day_starts = trade_price_plot.groupby('day')['session_time'].min().sort_index()
trade_products = sorted(trade_price_plot['product'].unique())

fig, axes = plt.subplots(len(trade_products), 1, figsize=(14, 8), sharex=True, constrained_layout=True)
axes = [axes] if len(trade_products) == 1 else axes
for ax, product in zip(axes, trade_products):
    product_data = trade_price_plot[trade_price_plot['product'] == product]
    sns.scatterplot(
        data=product_data,
        x='session_time',
        y='price',
        size='quantity',
        sizes=(30, 180),
        alpha=0.8,
        ax=ax,
    )
    for boundary in trade_day_starts.iloc[1:]:
        ax.axvline(boundary, color='0.75', linestyle='--', linewidth=1)
    ax.set_title(f'{product}: trade prices vs time')
    ax.set_ylabel('Trade price')
    ax.legend(title='Quantity')

axes[-1].set_xticks(trade_day_starts.tolist())
axes[-1].set_xticklabels([f'day {day}' for day in trade_day_starts.index])
axes[-1].set_xlabel('Trading session time')

trade_figure_path = FIGURES_DIR / 'trade-prices-vs-time-separate-assets.png'
fig.savefig(trade_figure_path, dpi=150, bbox_inches='tight')
plt.show()
trade_figure_path

In [ ]:
# Compare spread and imbalance distributions across products
spread_imbalance_plot = prices[['product', 'spread', 'book_imbalance']].dropna().copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.boxplot(data=spread_imbalance_plot, x='product', y='spread', ax=axes[0])
axes[0].set_title('Spread distribution by product')
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=spread_imbalance_plot, x='product', y='book_imbalance', ax=axes[1])
axes[1].set_title('Book imbalance distribution by product')
axes[1].tick_params(axis='x', rotation=15)

spread_imbalance_figure_path = FIGURES_DIR / 'spread-and-imbalance-distributions.png'
fig.savefig(spread_imbalance_figure_path, dpi=150, bbox_inches='tight')
plt.show()
spread_imbalance_figure_path

In [ ]:
# Join trades to the nearest prior quote snapshot so we can compare trade prices to candidate fair values
quote_reference = (
    prices[['day', 'timestamp', 'product', 'book_mid', 'wall_mid', 'spread', 'book_imbalance']]
    .sort_values(['product', 'day', 'timestamp'])
    .reset_index(drop=True)
)
trade_reference = (
    trades.rename(columns={'symbol': 'product'})
    .sort_values(['product', 'day', 'timestamp'])
    .reset_index(drop=True)
)

trade_with_quotes_parts = []
for (product, day), trade_group in trade_reference.groupby(['product', 'day'], sort=False):
    quote_group = quote_reference[
        (quote_reference['product'] == product) & (quote_reference['day'] == day)
    ][['timestamp', 'book_mid', 'wall_mid', 'spread', 'book_imbalance']]
    if quote_group.empty:
        continue

    aligned_group = pd.merge_asof(
        trade_group.sort_values('timestamp'),
        quote_group.sort_values('timestamp'),
        on='timestamp',
        direction='backward',
    )
    trade_with_quotes_parts.append(aligned_group)

trade_with_quotes = pd.concat(trade_with_quotes_parts, ignore_index=True)
trade_with_quotes['abs_trade_to_book_mid'] = (trade_with_quotes['price'] - trade_with_quotes['book_mid']).abs()
trade_with_quotes['abs_trade_to_wall_mid'] = (trade_with_quotes['price'] - trade_with_quotes['wall_mid']).abs()

trade_alignment_summary = (
    trade_with_quotes.groupby('product')
    .agg(
        trades_with_quote_match=('price', 'size'),
        mean_abs_trade_to_book_mid=('abs_trade_to_book_mid', 'mean'),
        mean_abs_trade_to_wall_mid=('abs_trade_to_wall_mid', 'mean'),
        median_abs_trade_to_book_mid=('abs_trade_to_book_mid', 'median'),
        median_abs_trade_to_wall_mid=('abs_trade_to_wall_mid', 'median'),
    )
    .sort_index()
)

display(trade_alignment_summary.round(3))

In [ ]:
# Fixed-fair diagnostic: test whether ASH_COATED_OSMIUM looks anchored around 10,000
resin_data = quote_price_plot[quote_price_plot['product'] == RESIN_CANDIDATE].copy()
resin_data['book_mid_minus_fair'] = resin_data['book_mid'] - RESIN_FAIR
resin_data['wall_mid_minus_fair'] = resin_data['wall_mid'] - RESIN_FAIR
resin_quotes = prices[prices['product'] == RESIN_CANDIDATE].copy()
resin_trades = trades[trades['symbol'] == RESIN_CANDIDATE].copy()

resin_summary = pd.DataFrame(
    {
        'metric': [
            'mean book mid',
            'std book mid',
            'share |book mid - fair| <= 2',
            'share ask <= fair',
            'share bid >= fair',
            'mean trade price',
            'share trade price within 2 ticks of fair',
        ],
        'value': [
            resin_quotes['book_mid'].mean(),
            resin_quotes['book_mid'].std(),
            (resin_quotes['book_mid'].sub(RESIN_FAIR).abs() <= 2).mean(),
            (resin_quotes['ask_price_1'] <= RESIN_FAIR).mean(),
            (resin_quotes['bid_price_1'] >= RESIN_FAIR).mean(),
            resin_trades['price'].mean(),
            (resin_trades['price'].sub(RESIN_FAIR).abs() <= 2).mean(),
        ],
    }
)

display(resin_summary.round(4))

fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
sns.lineplot(data=resin_data, x='session_time', y='book_mid_minus_fair', label='book mid - fair', ax=ax)
sns.lineplot(data=resin_data, x='session_time', y='wall_mid_minus_fair', label='wall mid - fair', ax=ax, alpha=0.75)
for boundary in day_starts.iloc[1:]:
    ax.axvline(boundary, color='0.75', linestyle='--', linewidth=1)
ax.axhline(0, color='black', linewidth=1)
ax.set_title(f'{RESIN_CANDIDATE}: deviation from the 10,000 fair anchor')
ax.set_ylabel('Ticks from fair')
ax.set_xlabel('Trading session time')

resin_anchor_figure_path = FIGURES_DIR / 'ash-coated-osmium-deviation-from-10000.png'
fig.savefig(resin_anchor_figure_path, dpi=150, bbox_inches='tight')
plt.show()
resin_anchor_figure_path

In [ ]:
# Dynamic-fair diagnostic: compare book mid and wall mid for INTARIAN_PEPPER_ROOT
kelp_data = quote_price_plot[quote_price_plot['product'] == KELP_CANDIDATE].copy()
kelp_trade_alignment = trade_with_quotes[trade_with_quotes['product'] == KELP_CANDIDATE].copy()

kelp_summary = pd.DataFrame(
    {
        'metric': [
            'book mid std',
            'wall mid std',
            'mean |wall - book|',
            'corr(book mid, wall mid)',
            'mean abs trade to book mid',
            'mean abs trade to wall mid',
        ],
        'value': [
            kelp_data['book_mid'].std(),
            kelp_data['wall_mid'].std(),
            kelp_data['wall_mid'].sub(kelp_data['book_mid']).abs().mean(),
            kelp_data[['book_mid', 'wall_mid']].corr().iloc[0, 1],
            kelp_trade_alignment['abs_trade_to_book_mid'].mean(),
            kelp_trade_alignment['abs_trade_to_wall_mid'].mean(),
        ],
    }
)

display(kelp_summary.round(4))

fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
sns.lineplot(data=kelp_data, x='session_time', y='book_mid', label='book_mid', ax=ax)
sns.lineplot(data=kelp_data, x='session_time', y='wall_mid', label='wall_mid', ax=ax, alpha=0.8)
for boundary in day_starts.iloc[1:]:
    ax.axvline(boundary, color='0.75', linestyle='--', linewidth=1)
ax.set_title(f'{KELP_CANDIDATE}: book mid vs wall mid')
ax.set_ylabel('Price')
ax.set_xlabel('Trading session time')

kelp_mid_figure_path = FIGURES_DIR / 'intarian-pepper-root-book-mid-vs-wall-mid.png'
fig.savefig(kelp_mid_figure_path, dpi=150, bbox_inches='tight')
plt.show()
kelp_mid_figure_path

In [ ]:
# Capture the current state in a small structure you can grow over time
round1_findings = {
    'products': sorted(prices['product'].unique().tolist()),
    'days_covered': sorted(prices['day'].unique().tolist()),
    'price_rows': int(len(prices)),
    'trade_rows': int(len(trades)),
    'raw_mid_zero_share': raw_mid_quality['raw_mid_zero_share'].round(4).to_dict(),
    'best_fixed_fair_candidate': RESIN_CANDIDATE,
    'fixed_fair_mean_book_mid': float(resin_quotes['book_mid'].mean()),
    'fixed_fair_mean_abs_deviation_from_10000': float(resin_quotes['book_mid'].sub(RESIN_FAIR).abs().mean()),
    'dynamic_fair_candidate': KELP_CANDIDATE,
    'dynamic_fair_mean_abs_wall_minus_book': float(kelp_data['wall_mid'].sub(kelp_data['book_mid']).abs().mean()),
    'trade_alignment_mean_abs_distance': trade_alignment_summary.round(3).to_dict(orient='index'),
}
round1_findings

## Results

Use the tables and plots above to decide:
- whether `ASH_COATED_OSMIUM` is stable enough to justify a fixed-fair market-making baseline around 10,000,
- whether `INTARIAN_PEPPER_ROOT` needs a book-derived fair estimate such as `wall_mid`,
- which spread, imbalance, and execution behaviours deserve follow-on feature work.

## Next Steps

- Turn the strongest fair-value proxy into a simple trader baseline for each product.
- Join trades to quote states more deeply if you want fill-quality or aggressor-side analysis.
- Add threshold sweeps for entry and exit rules once the fair-value definition feels stable.
